# Score-based models & SDEs — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normal_pdf(x,mu,sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
def softmax(a):
    a=np.asarray(a,dtype=float); e=np.exp(a-a.max()); return e/e.sum()
def mse(a,b): return float(np.mean((np.asarray(a)-np.asarray(b))**2))
print('setup ok')

## learning gradients of log density
Energy gradients, Gaussian noise, and stochastic differential equations are the prerequisites. The score $
abla_x\log p_t(x)$ points toward higher density at each noise level, becoming the vector field used by diffusion (10.12) and classifier-free guidance (10.13).

In [ ]:
# 1) Gaussian score points toward high density
x=np.linspace(-2,2,100); sigma=.5; score=-x/sigma**2
plt.figure(figsize=(5,3)); plt.plot(x,score); plt.axhline(0,c='k'); plt.title('score of N(0, sigma^2)')
assert score[0]>0 and score[-1]<0

In [ ]:
# 2) a denoising step follows the score inward
pts=np.array([-1.5,-.5,.5,1.5]); step=pts+0.1*(-pts/sigma**2)
plt.figure(figsize=(5,3)); plt.plot(pts,np.zeros_like(pts),'o',label='noisy'); plt.plot(step,np.zeros_like(step)+.1,'o',label='after score step'); plt.legend(); plt.title('score step denoises')
assert np.mean(np.abs(step))<np.mean(np.abs(pts))

In [ ]:
# 3) larger noise has gentler scores
sigmas=[.3,.7,1.2]; x=np.linspace(-2,2,100)
plt.figure(figsize=(5,3))
for s in sigmas: plt.plot(x,-x/s**2,label=f'sigma={s}')
plt.legend(); plt.title('score scale depends on noise level')
assert abs((-1)/sigmas[0]**2)>abs((-1)/sigmas[-1]**2)

In [ ]:
# 4) score field for a 2D Gaussian points to center
xx,yy=np.meshgrid(np.linspace(-2,2,9),np.linspace(-2,2,9)); u=-xx; v=-yy
plt.figure(figsize=(4,4)); plt.quiver(xx,yy,u,v); plt.title('2D score field')
assert u[0,0]>0 and v[0,0]>0

In [ ]:
# 5) annealed denoising uses a schedule of scores
x=2.0; path=[]
for s in [1.5,1.0,.7,.5,.3]: x=x+0.08*(-x/s**2); path.append(x)
plt.figure(figsize=(5,3)); plt.plot(path,'o-'); plt.title('noise schedule moves sample inward')
assert abs(path[-1])<2.0